# 03d · Selección del modelo por artículo (experimental)

**Propósito:** elegir para cada artículo el modelo que pronostica su ventana de backtesting, entre el seasonal naive, SARIMAX y CatBoost, con el filtro de la constitución, y medir si el conjunto elegido le gana al seasonal naive.

**Contexto:** el nb 03b trata los quiebres por tienda, arma los datasets, calcula el seasonal naive y hace la búsqueda SARIMAX (con su backtesting de 13 folds). El nb 03c busca los hiperparámetros del boosting y guarda el pronóstico de la ventana de la mejor configuración. Este notebook solo junta esos pronósticos y elige; no entrena nada.

**Alcance:** independiente y solo de experimentación. Persiste solo la tabla de selección (D1); el nb 04 no carga nada de aquí. Lo guardado por el 03b y el 03c se lee y no se modifica.

**Etapas** (el estado de cada una vive en `specs/001-abc-xyz-share-forecast/nb03d-etapas.md`):

| # | Etapa |
|---|---|
| 0 | Decisiones y reglas |
| 1 | Carga de los pronósticos de la ventana y de los pesos |
| 2 | Error por candidato (por artículo y agregado) |
| 3 | Selección por artículo y aceptación agregada |
| 4 | Diagnóstico de optimismo de la selección |
| 5 | Guardado de la tabla de selección y cierre |

## Etapa 0 · Decisiones y reglas

**Objetivo:** fijar qué hace este notebook y con qué reglas, antes de escribir código.

**Decisiones (usuario, 2026-11-07):**

| # | Decisión | Justificación |
|---|---|---|
| D1 | Independiente y experimental. Lee lo que guardaron el nb 03b (`data/interim/nb03b_datasets/` y los lotes de `data/interim/nb03b_sarimax/`) y el nb 03c (`data/interim/nb03c_boosting/catboost_backtest.pkl`). Solo persiste la tabla de selección en `data/interim/nb03d_selection/selection.csv`, una fila por artículo. El nb 04 no carga nada de aquí. | No repetir ningún cálculo (R5). El CSV deja la decisión por artículo a mano para el informe sin depender de este kernel. |
| D2 | **Candidatos:** seasonal naive (línea base obligatoria), SARIMAX (511 de los 517 artículos tienen modelo) y **CatBoost** (mejor WRMSSE del 03c: 0,506). XGBoost (0,507) y LightGBM (0,511) no compiten. Los 6 artículos sin SARIMAX eligen entre CatBoost y el seasonal naive. | Los tres boosting quedaron casi empatados. Si compitieran los tres, cada artículo elegiría el mínimo entre más candidatos medidos en los mismos 13 bloques, y ese mínimo sale más optimista por puro ruido (usuario). |
| D3 | **Error por artículo = RMSSE en sus 13 bloques de la ventana** (folds 1–13, h = 1), con la escala del 03b (`rmsse_scale`). **Filtro (constitución, Principio VII):** un candidato compite solo si su RMSSE es estrictamente menor que el del seasonal naive en esos mismos bloques. Entre los que pasan gana el de menor RMSSE; si ninguno pasa, el artículo usa el seasonal naive. Si hay empate exacto, gana el más simple (seasonal naive, luego SARIMAX, luego CatBoost). | Dentro de un artículo todos los candidatos se dividen por la misma escala, así que el RMSSE ordena igual que el RMSE en unidades. El empate exacto es casi imposible con valores continuos; la regla solo evita que el resultado dependa del orden de las columnas. |
| D4 | **Aceptación agregada:** el WRMSSE del conjunto elegido debe ser menor que el del seasonal naive (0,894), con los pesos del 03c (gasto en los últimos 13 bloques de train de cada artículo). Se reporta por segmento (AX, AY, AZ). **WAPE** en unidades como métrica secundaria (lectura de negocio: % de lo vendido) y **MAE** como diagnóstico. | Constitución, Principio VII. Mismos pesos y escalas que en el 03c para que los números se puedan comparar con lo ya visto. |
| D5 | **Diagnóstico de optimismo** (usuario): se elige con los folds 1–7 y se mide lo elegido en los folds 8–13, que no participaron en la elección. Se compara con el error de los mismos folds 8–13 si se eligiera mirándolos. No cambia la selección final, que usa los 13 folds. | Elegir el mínimo entre candidatos con los mismos bloques con que se mide da un error optimista. El corte es cronológico (los folds 8–13 son posteriores) y reparte los 13 folds en 7 para elegir y 6 para medir, sin solaparse. |
| D6 | Cómputo liviano: solo se reordenan pronósticos guardados (~6.700 filas por candidato). Un proceso, sin hilos extra. | No compite con nada por la CPU ni por la RAM. |

**Límites conocidos (se anotan junto a los resultados):**
- Los hiperparámetros de CatBoost se eligieron con los mismos 13 folds en los que compite (03c), mientras SARIMAX elige sus órdenes antes de la ventana: la comparación favorece algo a CatBoost.
- La selección por artículo también se hace con esos 13 folds, así que el WRMSSE del conjunto elegido es optimista. D5 mide cuánto. La medida limpia es el test (fuera de este notebook).
- En 231 de los 6.643 pronósticos de SARIMAX el reajuste del fold no convergió y se pronosticó con los últimos coeficientes que sí convergieron (respaldo del 03b, sin usar el bloque que se predice); se cuentan por artículo en la Etapa 1.

**Reglas que se mantienen:** R1–R8 de `CLAUDE.md`: una etapa a la vez, celdas md → import → run → check, todo número justificado, sin fuga (el test no entra).

## Configuración

**Qué hace:** fija la raíz del proyecto (rutas relativas), activa `%autoreload` para tomar los cambios de `src/` sin reiniciar el kernel, carga la configuración validada, fija la semilla e imprime las versiones (R6).
**Decisión:** no se carga ninguna librería de modelos: los pronósticos de SARIMAX y CatBoost llegan como tablas ya guardadas.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import platform
import sys
from importlib.metadata import version
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/.
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.config import load_validated_config

pd.set_option("display.max_columns", 30)

cfg = load_validated_config()
SEED = cfg["features"]["seed"]

LIBS = ["numpy", "pandas", "pyarrow", "matplotlib"]
display(pd.Series({"python": platform.python_version(), **{lib: version(lib) for lib in LIBS}}, name="versión").to_frame())
print(f"semilla: {SEED}")

,versión
python,3.12.3
numpy,2.5.4
pandas,3.1.0
pyarrow,26.0.0
matplotlib,3.11.2


semilla: 42


## Etapa 1 · Carga de los pronósticos de la ventana y de los pesos

**Objetivo:** tener en una sola tabla, por artículo y bloque de la ventana de backtesting (folds 1–13), las unidades reales y el pronóstico de los tres candidatos, y los pesos del WRMSSE.
**Entradas:**
- `data/interim/nb03b_datasets/` (nb 03b, 13.3): `snaive_valid` (seasonal naive en la ventana), `rmsse_scale` (escala del RMSSE por artículo) y `boosting_data` (fechas de los bloques, para los pesos).
- `data/interim/nb03b_sarimax/` (nb 03b, 14.3): lotes de la búsqueda SARIMAX, con el backtesting de 13 folds y el estado de cada artículo.
- `data/interim/nb03c_boosting/catboost_backtest.pkl` (nb 03c, 5.3): pronóstico de la ventana de la mejor configuración de CatBoost.
- Gasto diario por artículo (`load_interim`, solo hasta el último día de train) y el calendario.

**Salidas:** `window` (una fila por artículo y fold, con `units`, `snaive`, `sarimax`, `catboost` y `cell`), `weights` (peso por artículo), `rmsse_scale` y `status` (estado SARIMAX por artículo).
**Decisión:**
- Se carga, no se recalcula (R5): los pronósticos, la escala y los folds son los del 03b y el 03c.
- **Pesos = gasto en los últimos 13 bloques de train de cada artículo**, la misma regla del 03c (su Etapa 2), ahora en `train_weight_windows` para no copiar la celda. Como chequeo, el WRMSSE de CatBoost con estos pesos y escalas debe dar exactamente el de su mejor prueba en el 03c.
- **Todos los pronósticos en unidades** (CatBoost: `pred` = unidades del bloque anterior × ratio pronosticado). Las unidades reales de cada fuente deben coincidir fila a fila con las del seasonal naive; si no, `window_forecasts` se detiene.
- Los artículos sin modelo SARIMAX quedan con `sarimax` vacío y en la selección compiten sin él (D2).

In [2]:
from src.config import train_end
from src.data import load_calendar, load_interim, load_tables
from src.features import BLOCK_DAYS
from src.metrics import spend_weights, train_weight_windows, wrmsse_frame
from src.modeling import load_sarimax_checkpoints, window_forecasts

DATASETS_DIR = Path("data/interim/nb03b_datasets")                   # exportado por el nb 03b, 13.3
SARIMAX_DIR = Path("data/interim/nb03b_sarimax")                     # lotes de la búsqueda SARIMAX del nb 03b, 14.3
CATBOOST_FILE = Path("data/interim/nb03c_boosting/catboost_backtest.pkl")   # mejor configuración del nb 03c, 5.3
WEIGHT_BLOCKS = 13   # pesos = gasto en los últimos 13 bloques de train, igual que en el nb 03c (Etapa 2)

In [3]:
inputs, manifest = load_tables(DATASETS_DIR)
sarimax = load_sarimax_checkpoints(SARIMAX_DIR)
catboost = pd.read_pickle(CATBOOST_FILE)

In [4]:
window = window_forecasts(inputs["snaive_valid"], {"sarimax": sarimax["backtest"],
                                                   "catboost": catboost["backtest"].rename(columns={"pred": "catboost"})})

In [5]:
boosting_data = inputs["boosting_data"]
cal = load_calendar(cfg)
day_of = pd.Series(cal["d"].str[2:].astype(int).to_numpy(), index=pd.to_datetime(cal["date"]))   # fecha → día d
weight_windows = train_weight_windows(boosting_data, day_of, WEIGHT_BLOCKS, BLOCK_DAYS)
interim = load_interim(cfg, max_day=train_end(cfg))   # gasto diario solo hasta el último día de train
weights = spend_weights(interim["item_spend"], interim["item_ids"], weight_windows)

In [6]:
CANDIDATES = ["snaive", "sarimax", "catboost"]
rmsse_scale = inputs["rmsse_scale"].set_index("item_id")["scale"]
N_FOLDS = manifest["n_folds"]
n_items = manifest["n_items"]
cell_of = window.groupby("item_id")["cell"].first().astype(str)
print(f"nb 03b exportado: {manifest['saved_at']} · SARIMAX: {len(sarimax['meta'])} lotes, último {sarimax['meta']['saved_at'].max()} · "
      f"CatBoost: prueba {catboost['meta']['trial']} de {catboost['meta']['n_trials']}, guardado {catboost['meta']['saved_at']}")

# Mismo conjunto en las tres fuentes.
assert manifest["seed"] == SEED and N_FOLDS == 13 and manifest["snaive_m"] == 13
items = set(rmsse_scale.index)
assert len(items) == n_items == window["item_id"].nunique() and set(window["item_id"]) == items
assert set(sarimax["choice"]["item_id"].astype(str)) == items and sarimax["choice"]["item_id"].is_unique
assert set(catboost["backtest"]["item_id"].astype(str)) == items and catboost["meta"]["algorithm"] == "catboost"
# Ventana: 13 folds por artículo, un bloque por fold.
assert len(window) == n_items * N_FOLDS and not window.duplicated(["item_id", "fold"]).any()
assert (window.groupby("item_id")["fold"].agg(lambda f: sorted(f) == list(range(1, N_FOLDS + 1)))).all()
# Pronósticos: seasonal naive y CatBoost completos; SARIMAX solo en los artículos con modelo.
status = sarimax["choice"].set_index(sarimax["choice"]["item_id"].astype(str))["status"]
with_model = window["item_id"].map(status).eq("ok")
assert window[["snaive", "catboost"]].notna().all().all()
assert window.loc[with_model, "sarimax"].notna().all() and window.loc[~with_model, "sarimax"].isna().all()
# Pesos: positivos, sin días de test, 13 bloques completos.
assert set(weights.index) == items and (weights > 0).all()
assert max(b for _, b in weight_windows.values()) <= train_end(cfg)
assert all(b - a + 1 == WEIGHT_BLOCKS * BLOCK_DAYS for a, b in weight_windows.values())
# Reproduce el 03c: el WRMSSE de CatBoost con estos pesos y escalas es el de su mejor prueba.
cb_wrmsse = wrmsse_frame(window, "catboost", rmsse_scale, weights)[0]
sn_wrmsse = wrmsse_frame(window, "snaive", rmsse_scale, weights)[0]
assert np.isclose(cb_wrmsse, catboost["meta"]["wrmsse"], rtol=0, atol=1e-9), (cb_wrmsse, catboost["meta"]["wrmsse"])
print(f"WRMSSE reproducido · CatBoost {cb_wrmsse:.4f} (03c: {catboost['meta']['wrmsse']:.4f}) · seasonal naive {sn_wrmsse:.4f}")

display(pd.DataFrame({
    "artículos con pronóstico": {c: window.loc[window[c].notna(), "item_id"].nunique() for c in CANDIDATES},
    "pronósticos": {c: int(window[c].notna().sum()) for c in CANDIDATES},
    "negativos": {c: int((window[c] < 0).sum()) for c in CANDIDATES},
}).rename_axis("candidato"))
display(status.value_counts().rename("artículos").rename_axis("estado SARIMAX").to_frame())
no_model = status[status != "ok"]
display(pd.DataFrame({"estado": no_model, "cell": cell_of[no_model.index], "peso": (weights[no_model.index] / weights.sum()).round(4)})
        .rename_axis("artículos sin SARIMAX"))
backups = sarimax["choice"].set_index(sarimax["choice"]["item_id"].astype(str))["backtest_backups"]
backups = backups[status == "ok"]
print(f"respaldos de SARIMAX en la ventana: {int(backups.sum())} en {int((backups > 0).sum())} artículos")
display(backups[backups > 0].value_counts().sort_index().rename("artículos").rename_axis("respaldos (de 13)").to_frame().T)
display(window.sample(5, random_state=SEED))

nb 03b exportado: 2026-11-07T08:25:38 · SARIMAX: 52 lotes, último 2026-11-07T14:22:51 · CatBoost: prueba 43 de 100, guardado 2026-11-07T14:35:13
WRMSSE reproducido · CatBoost 0.5062 (03c: 0.5062) · seasonal naive 0.8940


,artículos con pronóstico,pronósticos,negativos
candidato,,,
snaive,517,6721,0
sarimax,511,6643,1
catboost,517,6721,0


,artículos
estado SARIMAX,
ok,511
sin_modelo,6


,estado,cell,peso
artículos sin SARIMAX,,,
FOODS_2_380,sin_modelo,AZ,0.0034
FOODS_3_087,sin_modelo,AX,0.0018
FOODS_3_250,sin_modelo,AY,0.0010
FOODS_3_499,sin_modelo,AX,0.0054
HOUSEHOLD_1_224,sin_modelo,AZ,0.0010
HOUSEHOLD_1_338,sin_modelo,AX,0.0018


respaldos de SARIMAX en la ventana: 231 en 170 artículos


respaldos (de 13),1,2,3,4
artículos,123,35,10,2


,item_id,block,fold,units,snaive,error,cell,sarimax,catboost
3552,FOODS_3_727,43,4,796.000000,830.000000,34.000000,AY,853.653452,892.213765
4108,HOBBIES_1_100,40,1,95.000000,123.000000,28.000000,AX,89.827633,105.551438
549,FOODS_2_022,43,4,490.500000,280.785714,-209.714286,AZ,385.276287,343.507930
1163,FOODS_2_322,46,7,791.000000,673.892857,-117.107143,AX,731.479904,707.036484
1881,FOODS_3_224,49,10,514.428571,763.000000,248.571429,AZ,373.987981,567.109133


## Etapa 2 · Error por candidato

**Objetivo:** medir cada candidato por artículo en los 13 bloques de su ventana y, agregado, por segmento (AX, AY, AZ) y en total, antes de elegir nada.
**Entradas:** `window`, `rmsse_scale`, `weights` y `cell_of` (Etapa 1).
**Salidas:** `errors` (una fila por artículo y candidato con `rmsse`, `wape`, `mae`, `n_blocks` y las sumas `abs_error` y `units`) y `summary` (WRMSSE y WAPE por candidato y segmento).
**Decisión:**
- **Por artículo** (D3, D4): RMSSE con la escala del 03b (la que usará la selección); WAPE = Σ|e| / Σy y MAE en unidades, en los mismos 13 bloques.
- **Agregado** (D4): WRMSSE con los pesos de la Etapa 1, renormalizados dentro de cada segmento; WAPE del grupo = Σ|e| / Σy de todos sus artículos y bloques (pondera por unidades, la lectura de negocio). El MAE queda solo por artículo: sumado entre artículos lo dominan los que más venden, y eso ya lo dice el WAPE.
- **Comparación sobre los mismos artículos:** la tabla principal usa los 511 artículos con los tres pronósticos. Si SARIMAX se midiera en sus 511 y los otros en 517, la diferencia mezclaría modelo y conjunto de artículos. Aparte se muestra el total de cada candidato en todos sus artículos (CatBoost y seasonal naive en 517 deben dar los 0,5062 y 0,8940 de la Etapa 1).
- Los pronósticos se miden tal como salieron del 03b y el 03c, sin recortar negativos (hay 1 de SARIMAX): recortarlo sería cambiar el modelo después de verlo.

In [7]:
from src.metrics import item_errors, segment_errors

SEGMENTS = ["AX", "AY", "AZ", "total"]

In [8]:
errors = item_errors(window, CANDIDATES, rmsse_scale)
common = set(errors.loc[errors["candidate"] == "sarimax", "item_id"])   # artículos con los tres candidatos
summary = segment_errors(errors[errors["item_id"].isin(common)], weights, cell_of)

In [9]:
# Una fila por artículo y candidato; SARIMAX solo en los artículos con modelo.
assert len(errors) == 2 * n_items + len(common) and not errors.duplicated(["item_id", "candidate"]).any()
assert len(common) == int((status == "ok").sum()) and (errors["n_blocks"] == N_FOLDS).all()
assert errors[["rmsse", "wape", "mae"]].notna().all().all() and (errors[["rmsse", "wape", "mae"]] >= 0).all().all()
# Agregado coherente con wrmsse_frame (Etapa 1): total de cada candidato en todos sus artículos.
own = segment_errors(errors, weights, cell_of).xs("total", level="segment")
assert np.isclose(own.loc["catboost", "wrmsse"], cb_wrmsse) and np.isclose(own.loc["snaive", "wrmsse"], sn_wrmsse)
assert np.isclose(own.loc["sarimax", "wrmsse"], wrmsse_frame(window[with_model], "sarimax", rmsse_scale, weights)[0])
assert (summary.xs("total", level="segment")["n_items"] == len(common)).all()

print(f"Comparación sobre los {len(common)} artículos con los tres candidatos:")
display(summary.unstack("candidate").reindex(SEGMENTS)[["n_items", "wrmsse", "wape"]]
        .reindex(columns=CANDIDATES, level="candidate").round(4))
print("Total de cada candidato en todos sus artículos:")
display(own.reindex(CANDIDATES).round(4))
print("RMSSE, WAPE y MAE por artículo (distribución):")
quantiles = errors.groupby("candidate")[["rmsse", "wape", "mae"]].quantile([0.1, 0.25, 0.5, 0.75, 0.9])
display(quantiles.unstack().reindex(CANDIDATES).round(3))
display(errors.sample(5, random_state=SEED))

Comparación sobre los 511 artículos con los tres candidatos:


n_items                   wrmsse                     wape          \
candidate  snaive sarimax catboost  snaive sarimax catboost  snaive sarimax   
segment                                                                       
AX            188     188      188  0.8693  0.7087   0.5523  0.1229  0.0919   
AY            228     228      228  0.8681  0.6396   0.4733  0.2482  0.1529   
AZ             95      95       95  1.0037  0.6335   0.4855  0.4270  0.1904   
total         511     511      511  0.8925  0.6640   0.5046  0.2477  0.1410   

                    
candidate catboost  
segment             
AX          0.0732  
AY          0.1070  
AZ          0.1356  
total       0.1023

Total de cada candidato en todos sus artículos:


,n_items,wrmsse,wape
candidate,,,
snaive,517,0.8940,0.2473
sarimax,511,0.6640,0.1410
catboost,517,0.5062,0.1023


RMSSE, WAPE y MAE por artículo (distribución):


rmsse                               wape                       \
            0.10   0.25   0.50   0.75   0.90   0.10   0.25   0.50   0.75   
candidate                                                                  
snaive     0.489  0.642  0.899  1.092  1.249  0.091  0.121  0.183  0.270   
sarimax    0.284  0.399  0.581  0.828  1.111  0.068  0.089  0.124  0.174   
catboost   0.229  0.323  0.453  0.614  0.808  0.057  0.073  0.095  0.125   

                     mae                                     
            0.90    0.10    0.25     0.50     0.75     0.90  
candidate                                                    
snaive     0.404  30.913  50.629  101.937  260.547  589.184  
sarimax    0.238  22.220  39.496   72.552  173.294  357.846  
catboost   0.163  18.214  29.245   52.544  129.760  253.419

,item_id,candidate,n_blocks,rmsse,wape,mae,abs_error,units
1495,HOUSEHOLD_2_037,sarimax,13,0.910871,0.108614,21.874714,284.371286,2618.178571
543,FOODS_3_349,snaive,13,0.612848,0.142036,155.401099,2020.214286,14223.250000
1268,HOUSEHOLD_1_277,sarimax,13,0.374532,0.144001,276.784559,3598.199261,24987.392857
528,FOODS_3_330,snaive,13,1.570021,0.320896,215.678571,2803.821429,8737.464286
1094,HOUSEHOLD_1_040,snaive,13,0.777081,0.161997,173.307692,2253.000000,13907.678571


## Etapa 3 · Selección por artículo y aceptación agregada

**Objetivo:** elegir el modelo de cada artículo y comprobar que el conjunto elegido le gana al seasonal naive.
**Entradas:** `errors` (Etapa 2), `window`, `rmsse_scale`, `weights` y `cell_of` (Etapa 1).
**Salidas:** `selection` (una fila por artículo con el RMSSE de cada candidato, `winner` y `winner_rmsse`), la columna `selected` en `window` (pronóstico del ganador) y `acceptance` (WRMSSE y WAPE del conjunto elegido por segmento).
**Decisión:**
- **Regla (D3, Principio VII):** por artículo, SARIMAX o CatBoost compiten solo si su RMSSE en los 13 folds es estrictamente menor que el del seasonal naive. Entre los que pasan gana el de menor RMSSE; si ninguno pasa, el seasonal naive. Empate exacto → el más simple, en el orden de `CANDIDATES` (seasonal naive, SARIMAX, CatBoost). Los 6 artículos sin SARIMAX eligen entre los otros dos.
- **Aceptación (D4):** WRMSSE del conjunto elegido < WRMSSE del seasonal naive, en los 517 artículos y con los pesos de la Etapa 1. Se reporta por segmento, con el WAPE como métrica secundaria.
- **Lectura del resultado:** el RMSSE del ganador es el mínimo de los candidatos en cada artículo, así que el WRMSSE del conjunto nunca supera al del mejor candidato único. Esa ventaja está medida con los mismos folds con los que se eligió, y por eso es optimista; la Etapa 4 mide cuánto.

In [10]:
from src.modeling import select_by_item, selected_forecast

In [11]:
selection = select_by_item(errors, CANDIDATES)
window["selected"] = selected_forecast(window, selection.set_index("item_id")["winner"])
acceptance = segment_errors(pd.concat([errors, item_errors(window, ["selected"], rmsse_scale)]), weights, cell_of)

In [12]:
sel = selection.set_index("item_id")
# Un ganador por artículo, con pronóstico; nunca peor que el seasonal naive ni que ningún candidato.
assert len(sel) == n_items and sel["winner"].isin(CANDIDATES).all() and window["selected"].notna().all()
assert (sel["winner_rmsse"] <= sel["snaive"]).all()
assert np.allclose(sel["winner_rmsse"], sel[CANDIDATES].min(axis=1))
not_base = sel["winner"] != "snaive"
assert (sel.loc[not_base, "winner_rmsse"] < sel.loc[not_base, "snaive"]).all()      # filtro estricto
assert (sel.loc[sel["sarimax"].isna(), "winner"] != "sarimax").all()                # sin SARIMAX no lo eligen
# El RMSSE del pronóstico armado coincide con el del ganador.
sel_rmsse = acceptance.xs(("selected", "total"))["wrmsse"]
assert np.isclose(sel_rmsse, wrmsse_frame(window, "selected", rmsse_scale, weights)[0])
assert np.isclose(sel_rmsse, np.dot(weights[sel.index] / weights.sum(), sel["winner_rmsse"]))
# Aceptación (D4).
base_rmsse = acceptance.xs(("snaive", "total"))["wrmsse"]
assert sel_rmsse < base_rmsse, (sel_rmsse, base_rmsse)
print(f"Aceptación: WRMSSE del conjunto elegido {sel_rmsse:.4f} < seasonal naive {base_rmsse:.4f} "
      f"({sel_rmsse / base_rmsse - 1:+.1%})")

# Ganadores: artículos y peso por segmento.
share = weights[sel.index] / weights.sum()
display(pd.concat({
    "artículos": pd.crosstab(cell_of[sel.index], sel["winner"], margins=True, margins_name="total"),
    "% del peso": pd.crosstab(cell_of[sel.index], sel["winner"], values=share, aggfunc="sum",
                              margins=True, margins_name="total").fillna(0).mul(100).round(1),
}, axis=1).reindex(columns=CANDIDATES + ["total"], level=1).rename_axis(index="segmento"))
# Conjunto elegido frente a cada candidato (517 artículos; SARIMAX en sus 511).
display(acceptance.unstack("candidate").reindex(SEGMENTS)[["wrmsse", "wape"]]
        .reindex(columns=CANDIDATES + ["selected"], level="candidate").round(4))
# Cuánto baja el RMSSE frente al seasonal naive en los artículos que no se quedan con él.
gain = (1 - sel.loc[not_base, "winner_rmsse"] / sel.loc[not_base, "snaive"]).rename("reducción del RMSSE vs. snaive")
display(gain.groupby(sel.loc[not_base, "winner"]).describe(percentiles=[0.1, 0.5, 0.9]).round(3))
display(selection.sample(5, random_state=SEED))

Aceptación: WRMSSE del conjunto elegido 0.4808 < seasonal naive 0.8940 (-46.2%)


artículos                        % del peso                        
winner      snaive sarimax catboost total     snaive sarimax catboost  total
segmento                                                                    
AX              15      25      151   191        4.3     5.3     27.6   37.2
AY              18      34      177   229        3.2     6.7     35.0   45.0
AZ               7      24       66    97        1.0     5.4     11.5   17.8
total           40      83      394   517        8.4    17.4     74.2  100.0

wrmsse                              wape                          
candidate  snaive sarimax catboost selected  snaive sarimax catboost selected
segment                                                                      
AX         0.8725  0.7087   0.5553   0.5265  0.1234  0.0919   0.0738   0.0698
AY         0.8685  0.6396   0.4747   0.4545  0.2479  0.1529   0.1070   0.1048
AZ         1.0034  0.6335   0.4832   0.4520  0.4281  0.1904   0.1358   0.1306
total      0.8940  0.6640   0.5062   0.4808  0.2473  0.1410   0.1023   0.0988

,count,mean,std,min,10%,50%,90%,max
winner,,,,,,,,
catboost,394.0,0.450,0.209,0.002,0.152,0.467,0.711,0.914
sarimax,83.0,0.526,0.194,0.105,0.210,0.548,0.760,0.881


,item_id,snaive,sarimax,catboost,winner,winner_rmsse
304,HOBBIES_1_040,0.694573,0.398229,0.337767,catboost,0.337767
501,HOUSEHOLD_2_135,0.967324,1.073322,0.591044,catboost,0.591044
441,HOUSEHOLD_1_339,0.836550,0.639697,0.342445,catboost,0.342445
153,FOODS_3_242,1.014372,0.488036,0.522370,sarimax,0.488036
503,HOUSEHOLD_2_176,1.003764,0.519228,0.479043,catboost,0.479043


## Etapa 4 · Diagnóstico de optimismo de la selección

**Objetivo:** saber si la ventaja de la selección por artículo (0,4808 frente a 0,5062 de CatBoost solo, Etapa 3) es real o sale de elegir y medir con los mismos folds.
**Entradas:** `window`, `rmsse_scale`, `weights`, `cell_of` y `selection` (Etapas 1–3).
**Salidas:** `late` (folds 8–13 con las columnas `honest` y `optimistic`), `diag` (WRMSSE y WAPE en los folds 8–13 por alternativa y segmento), `boot` (intervalos bootstrap) y `stability` (ganador con folds 1–7 frente al de los 13).
**Decisión (D5):**
- **Corte cronológico 7 / 6:** se elige con los folds 1–7 (misma regla de la Etapa 3) y se mide en los 8–13, que no participan en la elección. Los folds 8–13 son posteriores, como lo será el test.
- **Alternativas medidas en los folds 8–13:** `honest` (ganador elegido con 1–7), `optimistic` (ganador elegido con los mismos 8–13, el sesgo de la Etapa 3), CatBoost para todos, SARIMAX y seasonal naive. Optimismo = `honest` − `optimistic`.
- **¿Aporta la selección?** Diferencia pareada `honest` − CatBoost con un **bootstrap por artículos** (2.000 réplicas: con colas del 2,5 % deja unas 50 réplicas por cola, suficiente para un intervalo de percentiles estable; semilla del proyecto). Si el intervalo cruza el 0, con 6 bloques no se distingue de CatBoost para todos. Igual para `honest` − seasonal naive (aceptación honesta).
- **Estabilidad:** cuántos artículos mantienen el ganador elegido con 1–7 en la selección final de 13 folds.
- **Límites:** con 7 bloques se elige con menos información que con 13, así que `honest` sale algo pesimista frente a la selección final. Los hiperparámetros de CatBoost se buscaron con los 13 folds (03c), así que CatBoost en 8–13 también trae algo de optimismo: la comparación favorece algo a CatBoost para todos. No cambia la selección final.

In [13]:
from src.metrics import paired_bootstrap_wrmsse

CHOOSE_FOLDS = range(1, 8)    # D5: se elige con los folds 1–7
EVAL_FOLDS = range(8, 14)     # D5: se mide en los folds 8–13
N_BOOT = 2000                 # réplicas del bootstrap: ~50 por cola al 95 %

In [14]:
early = window[window["fold"].isin(CHOOSE_FOLDS)]
late = window[window["fold"].isin(EVAL_FOLDS)].copy()
choose = select_by_item(item_errors(early, CANDIDATES, rmsse_scale), CANDIDATES).set_index("item_id")
oracle = select_by_item(item_errors(late, CANDIDATES, rmsse_scale), CANDIDATES).set_index("item_id")
late["honest"] = selected_forecast(late, choose["winner"])
late["optimistic"] = selected_forecast(late, oracle["winner"])
late_errors = item_errors(late, CANDIDATES + ["honest", "optimistic"], rmsse_scale)
diag = segment_errors(late_errors, weights, cell_of)
late_rmsse = late_errors.pivot(index="item_id", columns="candidate", values="rmsse")
boot = pd.DataFrame({f"honest − {b}": paired_bootstrap_wrmsse(late_rmsse["honest"], late_rmsse[b], weights, N_BOOT, SEED)
                     for b in ["catboost", "snaive"]}).T

In [15]:
# Folds: 7 para elegir y 6 para medir, sin solaparse.
assert set(CHOOSE_FOLDS).isdisjoint(EVAL_FOLDS) and set(CHOOSE_FOLDS) | set(EVAL_FOLDS) == set(range(1, N_FOLDS + 1))
assert len(early) == n_items * 7 and len(late) == n_items * 6
assert late[["honest", "optimistic"]].notna().all().all()
# `optimistic` es el mínimo en 8–13: nunca peor que `honest` ni que el seasonal naive, artículo por artículo.
assert (late_rmsse["optimistic"] <= late_rmsse["honest"] + 1e-12).all()
assert np.allclose(late_rmsse["optimistic"], late_rmsse[CANDIDATES].min(axis=1))
assert np.isclose(diag.xs(("honest", "total"))["wrmsse"], wrmsse_frame(late, "honest", rmsse_scale, weights)[0])

wr = diag.xs("total", level="segment")["wrmsse"]
print(f"Folds 8–13 · WRMSSE: honesto {wr['honest']:.4f} · optimista {wr['optimistic']:.4f} "
      f"(optimismo {wr['honest'] - wr['optimistic']:.4f}) · CatBoost {wr['catboost']:.4f} · snaive {wr['snaive']:.4f}")
display(diag.unstack("candidate").reindex(SEGMENTS)[["wrmsse", "wape"]]
        .reindex(columns=["snaive", "sarimax", "catboost", "honest", "optimistic"], level="candidate").round(4))
print(f"Bootstrap pareado por artículos ({N_BOOT} réplicas, IC 95 %); diferencia < 0 = `honest` mejor:")
display(boot.round(4))

# Estabilidad: ganador con folds 1–7 frente al de la selección final (13 folds).
final = selection.set_index("item_id")["winner"]
stability = pd.crosstab(choose["winner"].rename("ganador 1–7"), final.rename("ganador 13 folds"), margins=True, margins_name="total")
display(stability.reindex(index=CANDIDATES + ["total"], columns=CANDIDATES + ["total"]).fillna(0).astype(int))
same = choose["winner"].eq(final[choose.index])
print(f"mismo ganador: {same.mean():.1%} de los artículos, {weights[same[same].index].sum() / weights.sum():.1%} del peso")
# De dónde sale la diferencia `honest` − CatBoost: aporte al WRMSSE según el ganador elegido con 1–7
# (los que eligieron CatBoost aportan 0 por construcción).
share_w = weights[late_rmsse.index] / weights.sum()
display(pd.DataFrame({
    "artículos": choose["winner"].value_counts(),
    "% del peso": share_w.groupby(choose["winner"]).sum().mul(100).round(1),
    "aporte a honest − catboost": ((late_rmsse["honest"] - late_rmsse["catboost"]) * share_w).groupby(choose["winner"]).sum().round(4),
}).reindex(CANDIDATES).rename_axis("ganador 1–7"))

Folds 8–13 · WRMSSE: honesto 0.5165 · optimista 0.4363 (optimismo 0.0802) · CatBoost 0.4831 · snaive 0.8265


wrmsse                                        wape          \
candidate  snaive sarimax catboost  honest optimistic  snaive sarimax   
segment                                                                 
AX         0.8183  0.6716   0.5472  0.5807     0.4772  0.1208  0.0897   
AY         0.7809  0.5996   0.4409  0.4711     0.4134  0.2220  0.1458   
AZ         0.9585  0.5974   0.4559  0.4974     0.4090  0.4720  0.2018   
total      0.8265  0.6257   0.4831  0.5165     0.4363  0.2439  0.1395   

                                       
candidate catboost  honest optimistic  
segment                                
AX          0.0771  0.0835     0.0675  
AY          0.1020  0.1117     0.0970  
AZ          0.1343  0.1639     0.1236  
total       0.1008  0.1138     0.0929

Bootstrap pareado por artículos (2000 réplicas, IC 95 %); diferencia < 0 = `honest` mejor:


,diff,low,high,p_a_not_better
honest − catboost,0.0334,0.0091,0.0560,0.996
honest − snaive,-0.3100,-0.3547,-0.2681,0.000


ganador 13 folds,snaive,sarimax,catboost,total
ganador 1–7,,,,
snaive,23,5,36,64
sarimax,3,63,40,106
catboost,14,15,318,347
total,40,83,394,517


mismo ganador: 78.1% de los artículos, 78.1% del peso


,artículos,% del peso,aporte a honest − catboost
ganador 1–7,,,
snaive,64,12.7,0.0253
sarimax,106,20.5,0.0082
catboost,347,66.7,0.0000


## Etapa 5 · Guardado de la tabla de selección y cierre

**Objetivo:** dejar la decisión por artículo en un CSV para el informe, sin depender de este kernel (D1).
**Entradas:** `selection` (Etapa 3), `choose` (Etapa 4), `errors`, `status`, `weights` y `cell_of`.
**Salidas:** `data/interim/nb03d_selection/selection.csv`, una fila por artículo:

| Columna | Contenido |
|---|---|
| `item_id`, `cell` | artículo y segmento |
| `weight` | peso del WRMSSE (fracción del total; Etapa 1) |
| `sarimax_status` | estado SARIMAX del 03b (`ok` o el motivo sin modelo) |
| `rmsse_snaive`, `rmsse_sarimax`, `rmsse_catboost` | RMSSE en los 13 folds (Etapa 2; vacío si no hay modelo) |
| `winner`, `winner_rmsse`, `winner_wape` | **selección de la constitución** (Principio VII, 13 folds; Etapa 3) |
| `winner_folds_1_7` | ganador al elegir solo con los folds 1–7 (diagnóstico de la Etapa 4) |
| `winner_catboost_all` | **alternativa sin selección: CatBoost para todos** (opción b del usuario) |

**Decisión:**
- **Opción b (usuario, 2026-11-08):** se guarda la selección vigente y, al lado, la alternativa CatBoost para todos, que fue mejor en la Etapa 4 (0,483 frente a 0,517 de la selección honesta). La regla vigente no cambia; la alternativa queda como evidencia para la enmienda pospuesta de la constitución.
- CSV simple, sin índice; se sobrescribe si existe (es la salida de este notebook y se reconstruye con él). Como chequeo se vuelve a leer y se compara con la tabla en memoria.

In [16]:
SELECTION_DIR = Path("data/interim/nb03d_selection")   # D1: única salida del notebook
SELECTION_FILE = SELECTION_DIR / "selection.csv"

In [17]:
sel = selection.set_index("item_id")
winner_wape = errors.set_index(["item_id", "candidate"])["wape"]
out = pd.DataFrame({
    "cell": cell_of[sel.index],
    "weight": weights[sel.index] / weights.sum(),
    "sarimax_status": status[sel.index],
    **{f"rmsse_{c}": sel[c] for c in CANDIDATES},
    "winner": sel["winner"],
    "winner_rmsse": sel["winner_rmsse"],
    "winner_wape": winner_wape.loc[list(zip(sel.index, sel["winner"]))].to_numpy(),
    "winner_folds_1_7": choose["winner"][sel.index],
    "winner_catboost_all": "catboost",
}).rename_axis("item_id").reset_index()
SELECTION_DIR.mkdir(parents=True, exist_ok=True)
out.to_csv(SELECTION_FILE, index=False)

In [18]:
back = pd.read_csv(SELECTION_FILE)
# Una fila por artículo, sin cambios al releer.
assert len(back) == n_items and back["item_id"].is_unique and set(back["item_id"]) == items
pd.testing.assert_frame_equal(back, out, check_dtype=False, check_categorical=False)
# Coherente con las etapas anteriores.
assert np.isclose(back["weight"].sum(), 1.0)
assert np.isclose(np.dot(back["weight"], back["winner_rmsse"]), sel_rmsse)              # WRMSSE de la Etapa 3
assert back.loc[back["sarimax_status"] != "ok", "rmsse_sarimax"].isna().all()
assert back["winner"].value_counts().to_dict() == selection["winner"].value_counts().to_dict()
print(f"guardado: {SELECTION_FILE} · {len(back)} artículos · {SELECTION_FILE.stat().st_size / 1024:.0f} KB")
display(back.head())

guardado: data/interim/nb03d_selection/selection.csv · 517 artículos · 83 KB


,item_id,cell,weight,sarimax_status,rmsse_snaive,rmsse_sarimax,rmsse_catboost,winner,winner_rmsse,winner_wape,winner_folds_1_7,winner_catboost_all
0,FOODS_1_004,AY,0.002776,ok,1.103393,0.556094,0.484460,catboost,0.484460,0.151948,sarimax,catboost
1,FOODS_1_012,AY,0.006406,ok,0.280223,0.151899,0.200179,sarimax,0.151899,0.041638,sarimax,catboost
2,FOODS_1_018,AX,0.001239,ok,0.552347,0.692752,0.487805,catboost,0.487805,0.097968,catboost,catboost
3,FOODS_1_023,AZ,0.002028,ok,0.449357,0.354513,0.166340,catboost,0.166340,0.059991,catboost,catboost
4,FOODS_1_024,AY,0.000842,ok,0.791394,0.670506,0.466422,catboost,0.466422,0.125532,catboost,catboost


### Cierre

- **Selección de la constitución (13 folds):** CatBoost en 394 artículos (74 % del peso), SARIMAX en 83 (17 %) y seasonal naive en 40 (8 %). Su WRMSSE, 0,481, cumple la aceptación frente al seasonal naive (0,894), en los tres segmentos.
- **Ese 0,481 es optimista:** se eligió y se midió con los mismos folds. Al elegir con los folds 1–7 y medir en los 8–13, la selección por artículo da 0,517 y CatBoost para todos 0,483: la selección pierde 0,033 (IC 95 % [0,009; 0,056]). La pérdida viene de artículos donde el seasonal naive o SARIMAX parecían mejores por azar.
- **Lectura:** con 13 bloques por artículo, elegir entre tres modelos agrega más ruido que señal; un único CatBoost global generaliza mejor. Ambas versiones quedan en `selection.csv` (`winner` y `winner_catboost_all`).
- **Pendiente fuera de este notebook:** usar CatBoost para todos sin el filtro por artículo exige enmendar el Principio VII, que se pospuso hasta terminar con los atípicos del nb 04. La medida limpia de cualquiera de las dos versiones es el test (nb 05).